# Temporal backtesting and error analysis

## Objective
Compare candidates at three rolling origins, then determine where errors occur by horizon, retail segment, and demand intermittency. A single aggregate WRMSSE is not enough to judge operational usefulness.

Origins `d_1857`, `d_1885`, and `d_1913` each evaluate the following 28 days. Training rows are strictly earlier than the fold origin, and hyperparameter validation uses an earlier origin again. Random cross-validation is rejected because it mixes future demand into training.


In [ ]:
profile = "dev"
run_id = "notebook-backtest"
force = False
execute_stage = False
seed = 42
wape_review_threshold = 0.50


## Metric contract

| Metric | Definition | Interpretation |
| --- | --- | --- |
| WRMSSE | Revenue-weighted RMSSE across all 12 M5 hierarchy levels | Primary M5 ranking; lower is better |
| WAPE | Sum of absolute error divided by sum of demand | Scale-free point error that remains defined with zero rows; lower is better |
| MAE | Mean absolute SKU-day error | Typical miss in units; lower is better |
| RMSE | Root mean squared SKU-day error | Penalizes large misses; lower is better |
| Bias | Sum of signed error divided by sum of demand | Negative means systematic under-forecasting |
| Coverage | Share of actuals inside q05 to q95 | A calibrated 90% interval should be close to 90% |
| Interval width | Mean q95 minus q05 | Coverage is less useful when achieved with very wide intervals |
| Fold degradation | Worst WRMSSE increase relative to seasonal naive | Temporal stability guardrail |

WRMSSE and WAPE answer different questions. A model can perform acceptably after hierarchy and revenue weighting while still making weak daily SKU-store predictions. Both must be reported.


In [ ]:
from IPython.display import display
import numpy as np
import pandas as pd

from retail_forecasting.config import load_config
from retail_forecasting.data.spark import get_spark, table_path
from retail_forecasting.forecasting.metrics import summarize_backtest_points
from retail_forecasting.forecasting.workflow import run_forecasting

config = load_config(profile)
execute_stage_enabled = str(execute_stage).strip().lower() in {"1", "true", "yes"}
stage_result = run_forecasting(config, run_id) if execute_stage_enabled else {"status": "reusing stored out-of-sample predictions"}
stage_result


In [ ]:
spark = get_spark(config, "notebook-error-analysis")
backtests = spark.read.format("delta").load(str(table_path(config, "gold", "backtest_forecasts"))).toPandas()
stored_metrics = spark.read.format("delta").load(str(table_path(config, "gold", "model_metrics"))).toPandas()
registered_forecast = spark.read.format("delta").load(str(table_path(config, "gold", "forecasts_bottom"))).select("model_name").first()
champion_name = registered_forecast["model_name"]
spark.stop()

backtests["abs_error"] = (backtests["yhat"] - backtests["target"]).abs()
backtests["signed_error"] = backtests["yhat"] - backtests["target"]
point_summary = summarize_backtest_points(backtests)
summary = stored_metrics.drop(columns=[column for column in point_summary.columns if column != "model_name"], errors="ignore").merge(point_summary, on="model_name", how="left")
display(summary.sort_values("mean_wrmsse"))


## Quality assessment
The review threshold below is deliberately separate from model promotion. Promotion currently uses WRMSSE, bias, interval coverage, and fold stability. WAPE is added as an explicit development warning so a formally eligible model is not described as operationally strong when bottom-level error remains high.


In [ ]:
forecast_models = backtests["model_name"].unique().tolist()
assert champion_name in forecast_models, f"Registered model {champion_name!r} is absent from backtests"
champion_row = summary.loc[summary["model_name"] == champion_name].iloc[0]
quality_assessment = pd.DataFrame([{
    "model": champion_name,
    "WAPE": champion_row["wape"],
    "review_threshold": wape_review_threshold,
    "status": "needs improvement" if champion_row["wape"] > wape_review_threshold else "passes review",
}])
display(quality_assessment)


## Zero demand and demand regimes
Demand regimes are defined using realized fold demand only for post-hoc diagnosis. They must not be joined back into training features. The bins reveal whether aggregate error is concentrated in intermittent products or remains high for dense series.


In [ ]:
reference = backtests.loc[backtests["model_name"] == champion_name]
activity = (
    reference.groupby("series_id")["target"]
    .agg(zero_rate=lambda values: float((values == 0).mean()), evaluated_demand="sum")
    .reset_index()
)
activity["demand_regime"] = pd.cut(
    activity["zero_rate"],
    bins=[-0.01, 0.50, 0.80, 0.95, 1.01],
    labels=["dense", "intermittent", "sparse", "very_sparse"],
)
diagnostic = backtests.merge(activity[["series_id", "zero_rate", "demand_regime"]], on="series_id", how="left")
regime_metrics = (
    diagnostic.groupby(["model_name", "demand_regime"], observed=True)
    .agg(rows=("target", "size"), series=("series_id", "nunique"), demand=("target", "sum"), abs_error=("abs_error", "sum"), signed_error=("signed_error", "sum"))
    .reset_index()
)
regime_metrics["wape"] = regime_metrics["abs_error"] / regime_metrics["demand"].clip(lower=1e-12)
regime_metrics["bias"] = regime_metrics["signed_error"] / regime_metrics["demand"].clip(lower=1e-12)
print(f"Zero-demand share across evaluated SKU-days: {(reference['target'] == 0).mean():.1%}")
display(regime_metrics.pivot(index="model_name", columns="demand_regime", values="wape").sort_index())


## Champion error by horizon and retail segment
Horizon analysis can expose decay or calendar-position effects. State and category analysis identifies business segments for which a global champion is not adequate.


In [ ]:
champion_predictions = diagnostic.loc[diagnostic["model_name"] == champion_name].copy()
horizon_metrics = champion_predictions.groupby("horizon").agg(
    demand=("target", "sum"), abs_error=("abs_error", "sum"), mae=("abs_error", "mean"), bias_units=("signed_error", "sum")
).reset_index()
horizon_metrics["wape"] = horizon_metrics["abs_error"] / horizon_metrics["demand"].clip(lower=1e-12)
horizon_metrics["bias"] = horizon_metrics["bias_units"] / horizon_metrics["demand"].clip(lower=1e-12)
display(horizon_metrics)
ax = horizon_metrics.plot(x="horizon", y="wape", marker="o", figsize=(11, 4), legend=False, title=f"{champion_name}: WAPE by forecast horizon")
ax.axhline(wape_review_threshold, color="#a94b45", linestyle="--", linewidth=1)
ax.set_ylabel("WAPE")
ax.grid(axis="y", alpha=0.25)


In [ ]:
segment_rows = []
for dimension in ["state_id", "cat_id", "dept_id"]:
    grouped = champion_predictions.groupby(dimension).agg(
        series=("series_id", "nunique"), demand=("target", "sum"), abs_error=("abs_error", "sum"), signed_error=("signed_error", "sum")
    ).reset_index().rename(columns={dimension: "segment"})
    grouped["dimension"] = dimension
    grouped["wape"] = grouped["abs_error"] / grouped["demand"].clip(lower=1e-12)
    grouped["bias"] = grouped["signed_error"] / grouped["demand"].clip(lower=1e-12)
    segment_rows.append(grouped)
segment_metrics = pd.concat(segment_rows, ignore_index=True)
display(segment_metrics[["dimension", "segment", "series", "demand", "wape", "bias"]].sort_values(["dimension", "wape"]))


## Interpretation of the current development run

The stored run should be described as a functioning benchmark, not as a production-quality forecast. More than half of evaluated SKU-days are zero, which makes bottom-level point forecasting difficult, but sparsity is not a sufficient explanation if dense-series WAPE also remains high.

The regime table also tests the assumption that one global champion is best everywhere. If different candidates lead in dense, intermittent, and very sparse groups, a demand-regime router or ensemble is a justified next experiment. It must be selected using training-period activity only and evaluated on untouched folds.


## Prioritized improvement experiments

1. Correct and version the forecast-boundary convention so the latest available sale is included consistently in lags and rolling windows.
2. Add lifecycle and price-relative variables: days since first sale, assortment age, price versus trailing item-store median, price change, and discount depth.
3. Evaluate Croston/TSB-style intermittent-demand baselines and a regime-aware model router.
4. Tune models against the primary hierarchy-aware objective or a closer weighted proxy rather than unweighted WAPE on a small random row sample.
5. Increase Optuna and N-HiTS budgets only after the data contract and feature gaps are fixed. More computation should not be used to hide a weak representation.
6. Add explicit WAPE and segment guardrails before describing any alias as operationally deployable.
